In [1]:
import pandas as pd

df = pd.read_csv("../../datasets/heart.csv")

required_columns = ["chol", "thalach"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

print("Dataset shape:", df.shape)
display(df[required_columns].head())

Dataset shape: (1025, 14)


,chol,thalach
0,212,168
1,203,155
2,174,125
3,203,161
4,294,106


In [2]:
def manual_percentile(values, percentile):
    values = sorted(float(x) for x in values if pd.notna(x))

    if len(values) == 0:
        return float("nan")

    position = (len(values) - 1) * percentile / 100
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    fraction = position - lower

    return values[lower] + fraction * (
        values[upper] - values[lower]
    )


def manual_min_max(value, minimum, maximum):
    if pd.isna(value):
        return float("nan")

    if maximum == minimum:
        return 0.0

    return (value - minimum) / (maximum - minimum)

In [3]:
df_featured = df.copy()

chol = pd.to_numeric(df_featured["chol"], errors="coerce")
thalach = pd.to_numeric(df_featured["thalach"], errors="coerce")

chol_min, chol_max = chol.min(), chol.max()
thalach_min, thalach_max = thalach.min(), thalach.max()

df_featured["chol_normalized"] = chol.apply(
    lambda x: manual_min_max(x, chol_min, chol_max)
)

df_featured["thalach_normalized"] = thalach.apply(
    lambda x: manual_min_max(x, thalach_min, thalach_max)
)

# Interaction term: normalized cholesterol × inverse normalized max heart rate
df_featured["interaction_term"] = (
    df_featured["chol_normalized"] *
    (1 - df_featured["thalach_normalized"])
)

# Percentile thresholds for cholesterol
q75 = manual_percentile(chol.dropna().tolist(), 75)
q90 = manual_percentile(chol.dropna().tolist(), 90)

def cholesterol_percentile_rank(value, values):
    if pd.isna(value) or len(values) == 0:
        return float("nan")

    count_below = sum(1 for x in values if x < value)
    count_equal = sum(1 for x in values if x == value)

    return (count_below + 0.5 * count_equal) / len(values) * 100

chol_values = chol.dropna().tolist()

df_featured["chol_percentile"] = chol.apply(
    lambda x: cholesterol_percentile_rank(x, chol_values)
)

# Example composite feature using domain-inspired terms
df_featured["risk_factor"] = (
    0.4 * df_featured["chol_normalized"]
    + 0.3 * (1 - df_featured["thalach_normalized"])
    + 0.3 * df_featured["interaction_term"]
)

df_featured["cholesterol_percentile_group"] = chol.apply(
    lambda x: (
        "Unknown" if pd.isna(x)
        else "High" if x >= q90
        else "Above 75th percentile" if x >= q75
        else "Below 75th percentile"
    )
)

print("Feature engineering completed.")
display(
    df_featured[
        [
            "chol",
            "thalach",
            "chol_normalized",
            "thalach_normalized",
            "interaction_term",
            "chol_percentile",
            "risk_factor",
            "cholesterol_percentile_group"
        ]
    ].head()
)

Feature engineering completed.


,chol,thalach,chol_normalized,thalach_normalized,interaction_term,chol_percentile,risk_factor,cholesterol_percentile_group
0,212,168,0.196347,0.740458,0.050960,26.731707,0.171689,Below 75th percentile
1,203,155,0.175799,0.641221,0.063073,18.243902,0.196875,Below 75th percentile
2,174,125,0.109589,0.412214,0.064415,4.585366,0.239496,Below 75th percentile
3,203,161,0.175799,0.687023,0.055021,18.243902,0.180719,Below 75th percentile
4,294,106,0.383562,0.267176,0.281083,83.219512,0.457597,Above 75th percentile


In [4]:
print("New feature summary:")
display(
    df_featured[
        [
            "interaction_term",
            "chol_percentile",
            "risk_factor"
        ]
    ].describe()
)

print("\nMissing values:")
print(
    df_featured[
        [
            "interaction_term",
            "chol_percentile",
            "risk_factor"
        ]
    ].isnull().sum()
)

New feature summary:


,interaction_term,chol_percentile,risk_factor
count,1025.000000,1025.000000,1025.000000
mean,0.111055,50.000000,0.264018
std,0.068710,28.880209,0.091451
min,0.000000,0.146341,0.066412
25%,0.061034,25.219512,0.196875
50%,0.096971,50.243902,0.254348
75%,0.143051,74.780488,0.319056
max,0.320611,99.853659,0.592366



Missing values:
interaction_term    0
chol_percentile     0
risk_factor         0
dtype: int64


In [5]:
output_path = "../../datasets/heart_feature_engineered.csv"

df_featured.to_csv(output_path, index=False)

print("Original shape:", df.shape)
print("Engineered shape:", df_featured.shape)
print("Saved:", output_path)

Original shape: (1025, 14)
Engineered shape: (1025, 20)
Saved: ../../datasets/heart_feature_engineered.csv


# Task 9: Feature Engineering Using Cholesterol and Thalach

## Objective
To create a new feature using cholesterol (`chol`) and maximum heart rate (`thalach`) with a composite formula, an interaction term, and percentile-based ranking.

## Methodology
1. Loaded the Heart Disease dataset.
2. Manually normalized cholesterol and maximum heart rate.
3. Created an interaction term using normalized cholesterol and inverse normalized maximum heart rate.
4. Calculated cholesterol percentile ranks using manual comparisons.
5. Created a composite `risk_factor` feature.
6. Grouped cholesterol observations according to percentile thresholds.
7. Saved the engineered dataset separately.

## Interpretation
The engineered features represent mathematical combinations of the available columns. They can help explore relationships in the dataset, but they are not medically validated risk scores.

## Conclusion
Feature engineering created additional numerical features and percentile-based categories from existing data. These features can be evaluated in future experiments to determine whether they improve predictive performance.
